# UniLumos BSS Official-Protocol Smoke

This notebook is Colab-first. It can clone or update the experiment branch from GitHub, mount Google Drive for weights and outputs, create the one-case smoke manifest, dry-run commands, and optionally run the smoke.

By default it does not run the model. Set `RUN_SMOKE = True` only after the path and weight checks pass.

## 1. Configure Sources And Drive Paths

Edit these values only if you move the fork, branch, weights, or output root.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess

GITHUB_REPO = "https://github.com/WANG-Ruipeng/Lumos-Custom.git"
BRANCH = "bss-unilumos-official-smoke"
LOCAL_REPO_ROOT = Path("/content/Lumos-Custom")

DRIVE_ROOT = Path("/content/drive/MyDrive")
WEIGHTS_ROOT = DRIVE_ROOT / "Colab_Projects" / "UniLumos" / "weights"
EXP_ROOT = DRIVE_ROOT / "Colab_Projects" / "UniLumos-BSS-Runs" / "model_b_unilumos_official_bss_smoke_v1"

RUN_INSTALL = False
RUN_SMOKE = False
RUN_METRICS = False

print("GITHUB_REPO =", GITHUB_REPO)
print("BRANCH =", BRANCH)
print("LOCAL_REPO_ROOT =", LOCAL_REPO_ROOT)
print("WEIGHTS_ROOT =", WEIGHTS_ROOT)
print("EXP_ROOT =", EXP_ROOT)

## 2. Mount Google Drive

Weights and experiment outputs live on Drive. Code is cloned into Colab's local `/content` workspace.

In [ ]:
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; Drive mount skipped.")

print("Drive available:", DRIVE_ROOT.exists())

## 3. Clone Or Pull The Experiment Branch

This cell is safe to rerun. If `/content/Lumos-Custom` already exists, it fetches and fast-forward pulls the configured branch. Otherwise it clones the branch.

In [ ]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if LOCAL_REPO_ROOT.exists() and (LOCAL_REPO_ROOT / ".git").exists():
    run(["git", "fetch", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
    run(["git", "checkout", BRANCH], cwd=LOCAL_REPO_ROOT)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
else:
    LOCAL_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "-b", BRANCH, GITHUB_REPO, str(LOCAL_REPO_ROOT)])

TASK_ROOT = LOCAL_REPO_ROOT / "UniLumos"
CODE_ROOT = TASK_ROOT / "UniLumos"
assert (CODE_ROOT / "unilumos_infer_abc.py").exists(), CODE_ROOT

os.environ["PYTHONPATH"] = f"{TASK_ROOT}:{CODE_ROOT}:" + os.environ.get("PYTHONPATH", "")
if str(TASK_ROOT) not in sys.path:
    sys.path.insert(0, str(TASK_ROOT))
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=LOCAL_REPO_ROOT, text=True).strip()
print("TASK_ROOT =", TASK_ROOT)
print("CODE_ROOT =", CODE_ROOT)
print("Checked out commit =", commit)

## 4. Optional Dependency Install

Set `RUN_INSTALL = True` in the first cell if this Colab runtime does not already have UniLumos dependencies. Keep it off when using a prepared runtime.

In [ ]:
if RUN_INSTALL:
    run([sys.executable, "-m", "pip", "install", "-r", str(CODE_ROOT / "requirements.txt")])
else:
    print("Dependency install skipped. Set RUN_INSTALL = True if needed.")

## 5. Verify Code, Examples, And Weights

The local repository contains examples. The large model files should be on Drive under `WEIGHTS_ROOT`.

In [ ]:
required_code = [
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_manifest_unilumos_official_smoke.py",
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py",
    CODE_ROOT / "examples" / "examples_refined.csv",
    CODE_ROOT / "unilumos_infer_abc.py",
    CODE_ROOT / "src" / "schedulers" / "RFLOW_WANX21_T2V.py",
]
required_weights = [
    WEIGHTS_ROOT / "models_t5_umt5-xxl-enc-bf16.pth",
    WEIGHTS_ROOT / "umt5-xxl",
    WEIGHTS_ROOT / "vae.pth",
    WEIGHTS_ROOT / "unilumos.pt",
]

missing = []
for path in required_code + required_weights:
    ok = path.exists()
    print(("OK      " if ok else "MISSING "), path)
    if not ok:
        missing.append(str(path))

if missing:
    print()
    print("Missing items detected. You can still run manifest dry-run, but do not set RUN_SMOKE=True until weights exist.")


## 6. Validate BSS Schedule Logic Locally

This is lightweight and does not load UniLumos weights.

In [ ]:
from bss_experiments.unilumos.bss_core.grids import get_uniform_shifted_sigmas, make_boundary_split_coords
from bss_experiments.unilumos.bss_core.validate import validate_schedule_payload

base8 = get_uniform_shifted_sigmas(8, 8.0)
bss10, meta = make_boundary_split_coords(base8)
payload = {
    "model_name": "UniLumos",
    "method": "bss10",
    "sampler_mode": "bss",
    "sample_steps": 10,
    "sample_shift": 8.0,
    "base_sample_steps": 8,
    "actual_nfe": 10,
    "split_pairs": "0,-1",
    "terminal_coord": 0.0,
    "base_sigmas": base8,
    "final_sigmas": bss10,
    "timesteps": list(range(10)),
}
result = validate_schedule_payload(payload, raise_on_error=True)
print(json.dumps(result, indent=2))

## 7. Create One-Case Smoke Manifest

The manifest uses the first official `abc` example and creates four rows: `uniform8`, `uniform10`, `bss10`, and `reference_uniform25`.

In [ ]:
manifest_cmd = [
    sys.executable,
    str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_manifest_unilumos_official_smoke.py"),
    "--experiment_root", str(EXP_ROOT),
    "--max_cases", "1",
    "--mode", "abc",
]
manifest_path = subprocess.check_output(manifest_cmd, text=True).strip()
print("manifest_path =", manifest_path)

## 8. Dry-Run Commands

This writes `reports/01_dry_run_commands.md` under `EXP_ROOT` and prints the exact commands. It does not run the model.

In [ ]:
dry_run_cmd = [
    sys.executable,
    str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
    "--manifest", manifest_path,
    "--weights_root", str(WEIGHTS_ROOT),
    "--dry_run",
]
subprocess.check_call(dry_run_cmd)
print("Dry-run report:", EXP_ROOT / "reports" / "01_dry_run_commands.md")

## 9. Run One-Case Smoke

Set `RUN_SMOKE = True` in the first cell only after the weight check passes. This runs exactly four rows.

In [ ]:
if RUN_SMOKE:
    run_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
        "--manifest", manifest_path,
        "--weights_root", str(WEIGHTS_ROOT),
        "--resume",
    ]
    subprocess.check_call(run_cmd)
else:
    print("RUN_SMOKE is False; model execution skipped.")

## 10. Validate Dumped Schedules

Run after the smoke has produced schedule JSON files.

In [ ]:
schedule_paths = sorted((EXP_ROOT / "schedules").glob("*_schedule.json"))
print("schedule json count:", len(schedule_paths))
if schedule_paths:
    validate_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "validate_schedules.py"),
        *map(str, schedule_paths),
    ]
    subprocess.check_call(validate_cmd)
else:
    print("No schedules found yet.")

## 11. Metrics, Table, And Side-By-Side

Set `RUN_METRICS = True` after the four smoke outputs exist.

In [ ]:
if RUN_METRICS:
    metrics_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "compute_metrics_against_ref.py"),
        "--manifest", manifest_path,
    ]
    metrics_csv = subprocess.check_output(metrics_cmd, text=True).strip()
    print("metrics_csv =", metrics_csv)

    table_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_main_table.py"),
        "--metrics_csv", metrics_csv,
    ]
    subprocess.check_call(table_cmd)

    side_by_side_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_side_by_side.py"),
        "--manifest", manifest_path,
    ]
    subprocess.check_call(side_by_side_cmd)
else:
    print("RUN_METRICS is False; metrics/table/side-by-side skipped.")

## 12. Sync / Update Notes

- Rerun the clone/pull cell to sync Colab code with the GitHub branch.
- Outputs are already written under `EXP_ROOT` on Google Drive, so they persist across Colab sessions.
- Do not commit generated videos or weights to GitHub.